In [ ]:
import math
import torch
from torch import nn
import torch.nn.functional as F
import functools
import numpy as np
import matplotlib.pyplot as plt
import os
from torchvision import transforms
from math import pi
import SimpleITK as sitk
import plotly
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import vtk
from vtk.util.numpy_support import vtk_to_numpy
import pickle
import monai
from monai.transforms import (        
    EnsureChannelFirst,
    RepeatChannel,
    Compose,    
    RandFlip,
    RandRotate,
    CenterSpatialCrop,
    ScaleIntensityRange,
    ScaleIntensity,
    ToTensor, 
    RandAdjustContrast,
    RandGaussianNoise,
    RandGaussianSmooth,
    BorderPad,
    RandSpatialCrop,
    NormalizeIntensity,
    RandAffined,
    EnsureChannelFirstd,
    LoadImaged,
    RandRotated,
    ToTensord
)

import pytorch_lightning as pl

# alpha_coeff_boundary_map = 0.1
# beta_coeff_scattering = 10  #100 approximates it closer
# TGC = 8
# CLAMP_VALS = True

def island_removal(x):
    """
    Assigns a label to unlabeled pixels in a batch of single-channel images based on the most common label among its neighbors.

    Args:
    images (Tensor): The input batch of single-channel image tensors with shape (N, C, H, W), C=1.
    num_classes (int): The number of different labels in the images, excluding the unlabeled (-1).

    Returns:
    Tensor: The batch of image tensors after label assignment, with shape (N, C, H, W).
    """
    num_classes = int(torch.max(x))
    # Squeeze out the channel dimension (C=1)
    x = x.squeeze(1)

    # One-hot encode the images
    one_hot = F.one_hot(x.to(torch.long), num_classes + 1).permute(0, 3, 1, 2).float()

    # Define the convolution kernel for counting neighbors
    kernel_size = 3
    conv_filter = torch.ones((num_classes + 1, 1, kernel_size, kernel_size), device=x.device)
    conv_filter[:, :, 1, 1] = 0 # Ignore the center pixel

    # Count the labels in the neighborhood of each pixel
    neighbor_counts = F.conv2d(one_hot, conv_filter, padding=1, groups=num_classes + 1)

    # Select the most frequent label (excluding the temporary label for unlabeled pixels)
    _, most_frequent_labels = torch.max(neighbor_counts[:, :-1, :, :], dim=1)

    # Replace the unlabeled pixels in the original images
    unlabeled_mask = (x == 0)
    result = x.where(~unlabeled_mask, most_frequent_labels)

    # Add the channel dimension back
    return result.unsqueeze(1)


def gaussian_kernel(size: int, mean: float, std: float):
    d1 = torch.distributions.Normal(mean, std)
    d2 = torch.distributions.Normal(mean, std*3)
    vals_x = d1.log_prob(torch.arange(-size, size+1, dtype=torch.float32)).exp()
    vals_y = d2.log_prob(torch.arange(-size, size+1, dtype=torch.float32)).exp()

    gauss_kernel = torch.einsum('i,j->ij', vals_x, vals_y)
    
    return gauss_kernel / torch.sum(gauss_kernel).reshape(1, 1)

g_kernel = gaussian_kernel(3, 0., 0.5)
g_kernel = torch.tensor(g_kernel[None, None, :, :])




In [ ]:
class UltrasoundRendering(pl.LightningModule):
    def __init__(self, **kwargs): #num_labels=340, w=256, h=256, center_x=128.0, center_y=-20.0, r1=40.0, r2=224.0, theta=np.pi/4
        super().__init__()

        self.save_hyperparameters()
        
        
        # df = pd.read_csv(acoustic_params_fn)        
        # accoustic_imped,attenuation,mu_0,mu_1,sigma_0
        self.acoustic_impedance_dict = torch.nn.Parameter(torch.rand(self.hparams.num_labels))    # Z in MRayl
        self.attenuation_dict =    torch.nn.Parameter(torch.rand(self.hparams.num_labels))   # alpha in dB cm^-1 at 1 MHz
        self.mu_0_dict =           torch.nn.Parameter(torch.rand(self.hparams.num_labels)) # mu_0 - scattering_mu   mean brightness
        self.mu_1_dict =           torch.nn.Parameter(torch.rand(self.hparams.num_labels)) # mu_1 - scattering density, Nr of scatterers/voxel
        self.sigma_0_dict =        torch.nn.Parameter(torch.rand(self.hparams.num_labels)) # sigma_0 - scattering_sigma - brightness std
        
        grid = self.compute_grid(self.hparams.grid_w, self.hparams.grid_h, self.hparams.center_x, self.hparams.center_y, self.hparams.r1, self.hparams.r2, self.hparams.theta)


        inverse_grid, self.mask_fan = self.compute_grid_inverse(grid)
        
        grid = grid / torch.tensor([self.hparams.grid_w, self.hparams.grid_h]) * 2.0 - 1.0
        inverse_grid = inverse_grid / torch.tensor([self.hparams.grid_w, self.hparams.grid_h]) * 2.0 - 1.0

        self.register_buffer("grid", grid)
        self.register_buffer("inverse_grid", inverse_grid)
        
        self.g_kernel = self.gaussian_kernel_asym(3, 0., 0.5)[None, None, :, :]

        self.loss = nn.L1Loss()

        
    def init_params(self, df):
        # df = pd.read_csv(acoustic_params_fn)
        
        # accoustic_imped,attenuation,mu_0,mu_1,sigma_0
        self.acoustic_impedance_dict = torch.nn.Parameter(torch.tensor(df['accoustic_imped'], dtype=torch.float32))    # Z in MRayl
        self.attenuation_dict =    torch.nn.Parameter(torch.tensor(df['attenuation'], dtype=torch.float32))   # alpha in dB cm^-1 at 1 MHz
        self.mu_0_dict =           torch.nn.Parameter(torch.tensor(df['mu_0'], dtype=torch.float32)) # mu_0 - scattering_mu   mean brightness
        self.mu_1_dict =           torch.nn.Parameter(torch.tensor(df['mu_1'], dtype=torch.float32)) # mu_1 - scattering density, Nr of scatterers/voxel
        self.sigma_0_dict =        torch.nn.Parameter(torch.tensor(df['sigma_0'], dtype=torch.float32)) # sigma_0 - scattering_sigma - brightness std

    def compute_grid(self, w, h, center_x, center_y, r1, r2, theta):
        # Convert inputs to tensors
        angles = torch.linspace(-theta, theta, w)  # Angles from -theta to theta
        radii = torch.linspace(r1, r2, h)  # Linear space of radii

        # Calculate sin and cos for all angles (broadcasting)
        sin_angles = torch.sin(angles)
        cos_angles = torch.cos(angles)

        # Initialize the grid for intersection points
        # Shape of grid: (h, w, 2) where 2 represents (x, y) coordinates
        grid = torch.zeros(h, w, 2)

        # Calculate intersections for each radius and angle
        for i, radius in enumerate(radii):
            x = (center_x + radius * sin_angles) # x coordinates for all angles at this radius
            y = (center_y + radius * cos_angles) # y coordinates for all angles at this radius

            grid[i] = torch.stack((x, y), dim=1)  # Update grid with coordinates

        return grid

    def compute_grid_inverse(self, grid):
        h, w, _ = grid.shape  # grid dimensions
        inverse_grid = torch.zeros(h, w, 2)  # Initialize inverse grid
        mask = torch.zeros(1, h, w)  # Initialize mask

        # Iterate through each point in the grid
        for j in range(h):
            for i in range(w):
                # Extract the polar coordinates (represented in the grid)
                xi, yi = torch.round(grid[j, i]).to(torch.long)

                # Place the Cartesian coordinates in the inverse grid
                if 0 <= xi and xi < w and 0 <= yi and yi < h:
                    inverse_grid[yi, xi] = torch.tensor([i, j])
                    mask[0, yi, xi] = 1

        return inverse_grid, mask
    
    def gaussian_kernel_sym(self, size, sigma):
        """
        Creates a 2D Gaussian kernel.

        Args:
        size (int): The size of the kernel.
        sigma (float): The standard deviation of the Gaussian distribution.

        Returns:
        Tensor: A 2D Gaussian kernel.
        """
        coords = torch.arange(size, dtype=torch.float32)
        coords -= size // 2

        g = coords**2
        g = (-g / (2 * sigma**2)).exp()

        g /= g.sum()
        
        gaussian = g.outer(g)
        
        return gaussian
    
    def gaussian_kernel_asym(self, size: int, mean: float, std: float):
        d1 = torch.distributions.Normal(mean, std)
        d2 = torch.distributions.Normal(mean, std*3)
        vals_x = d1.log_prob(torch.arange(-size, size+1, dtype=torch.float32)).exp()
        vals_y = d2.log_prob(torch.arange(-size, size+1, dtype=torch.float32)).exp()

        gauss_kernel = torch.einsum('i,j->ij', vals_x, vals_y)

        return gauss_kernel / torch.sum(gauss_kernel).reshape(1, 1)


    def smooth(self, x, kernel_size=3, sigma=1, maxpool=True):
        """
        Applies Gaussian blur to a batch of images.

        Args:
        images (Tensor): Input images with shape (N, C, H, W).
        kernel_size (int): The size of the Gaussian kernel.
        sigma (float): The standard deviation of the Gaussian distribution.

        Returns:
        Tensor: The smoothed images.
        """
        # Create a Gaussian kernel
        kernel = self.gaussian_kernel_sym(kernel_size, sigma).to(x.device)
        
        kernel = kernel.expand(x.size(1), 1, kernel_size, kernel_size)

        # Apply the Gaussian kernel to each image in the batch
        padding = kernel_size // 2
        
        if maxpool:
            x = torch.nn.MaxPool2d(kernel_size, stride=1, padding=1)(x)
            
        return F.conv2d(x, kernel, padding=padding, groups=x.size(1))
    
    def rendering(self, shape, attenuation_medium_map, mu_0_map, mu_1_map, sigma_0_map, z_vals=None, refl_map=None, boundary_map=None):
        
        dists = torch.abs(z_vals[..., :-1, None] - z_vals[..., 1:, None])     # dists.shape=(W, H-1, 1)
        dists = dists.squeeze(-1)                                             # dists.shape=(W, H-1)
        dists = torch.cat([dists, dists[:, -1, None]], dim=-1)                # dists.shape=(W, H)

        attenuation = torch.exp(-attenuation_medium_map * dists)
        attenuation_total = torch.cumprod(attenuation, dim=3, dtype=torch.float32, out=None)

        gain_coeffs = torch.linspace(1, self.hparams.tgc, attenuation_total.shape[3], device=self.device)
        gain_coeffs = torch.tile(gain_coeffs, (attenuation_total.shape[2], 1))
        attenuation_total = attenuation_total * gain_coeffs     # apply TGC

        reflection_total = torch.cumprod(1. - refl_map * boundary_map, dim=3, dtype=torch.float32, out=None) 
        reflection_total = reflection_total.squeeze(-1) 
        reflection_total_plot = torch.log(reflection_total + torch.finfo(torch.float32).eps)

        texture_noise = torch.randn(shape, dtype=torch.float32, device=self.device)
        scattering_probability = torch.randn(shape, dtype=torch.float32, device=self.device)

        # scattering_zero = torch.zeros(shape, dtype=torch.float32)

        z = mu_1_map - scattering_probability
        sigmoid_map = torch.sigmoid(self.hparams.beta_coeff_scattering * z)

        # approximating  Eq. (4) to be differentiable:
        # where(scattering_probability <= mu_1_map, 
        #                     texture_noise * sigma_0_map + mu_0_map, 
        #                     scattering_zero)
        # scatterers_map =  (sigmoid_map) * (texture_noise * sigma_0_map + mu_0_map) + (1 -sigmoid_map) * scattering_zero   # Eq. (6)
        scatterers_map =  (sigmoid_map) * (texture_noise * sigma_0_map + mu_0_map)

        psf_scatter_conv = torch.nn.functional.conv2d(input=scatterers_map, weight=self.g_kernel, stride=1, padding="same")
        # psf_scatter_conv = psf_scatter_conv.squeeze()

        b = attenuation_total * psf_scatter_conv    # Eq. (3)

        border_convolution = torch.nn.functional.conv2d(input=boundary_map, weight=self.g_kernel, stride=1, padding="same")
        # border_convolution = border_convolution.squeeze()

        r = attenuation_total * reflection_total * refl_map * border_convolution # Eq. (2)
        
        intensity_map = b + r   # Eq. (1)
        # intensity_map = intensity_map.squeeze() 
        intensity_map = torch.clamp(intensity_map, 0, 1)

        return intensity_map, attenuation_total, reflection_total_plot, scatterers_map, scattering_probability, border_convolution, texture_noise, b, r
    
    def render_rays(self, W, H):
        N_rays = W 
        t_vals = torch.linspace(0., 1., H, device=self.device)
        z_vals = t_vals.unsqueeze(0).expand(N_rays , -1) * 4 

        return z_vals

    def forward(self, x):
        #init tissue maps
        #generate maps from the dictionary and the input label map
        repeats = [1,]*len(x.shape)
        repeats[0] = x.shape[0]

        #UNWARP
        grid = self.grid.repeat(repeats)
        
        x = F.grid_sample(x.float(), grid, mode='nearest', padding_mode='zeros', align_corners=True)

        x = torch.rot90(x, k=1, dims=[2, 3])
        x = x.to(torch.long)
        
        acoustic_imped_map = self.acoustic_impedance_dict[x]
        attenuation_medium_map = self.attenuation_dict[x]
        mu_0_map = self.mu_0_dict[x]
        mu_1_map = self.mu_1_dict[x]
        sigma_0_map = self.sigma_0_dict[x]

        
        #Comput the difference along dimension 2
        diff_arr = torch.diff(acoustic_imped_map, dim=2)                
        # The pad tuple is (padding_left,padding_right, padding_top,padding_bottom)
        # The array is padded at the top
        diff_arr = F.pad(diff_arr, (0,0,1,0))

        #Compute the boundary map using the diff_array
        boundary_map =  -torch.exp(-(diff_arr**2)/self.hparams.alpha_coeff_boundary_map) + 1
        
        #Roll/shift the elements along dimension 2 and set the last element to 0
        shifted_arr = torch.roll(acoustic_imped_map, -1, dims=2)
        shifted_arr[-1:] = 0

        # This computes the sum/accumulation along the direction and set elements that are 0 to 1. Compute the division
        sum_arr = acoustic_imped_map + shifted_arr
        sum_arr[sum_arr == 0] = 1
        div = diff_arr / sum_arr
        # Compute the reflection from the elements
        refl_map = div ** 2
        refl_map = torch.sigmoid(refl_map)      # 1 / (1 + (-refl_map).exp())

        z_vals = self.render_rays(x.shape[2], x.shape[3])

        # if CLAMP_VALS:
        #     attenuation_medium_map = torch.clamp(attenuation_medium_map, 0, 10)
        #     acoustic_imped_map = torch.clamp(acoustic_imped_map, 0, 10)
        #     sigma_0_map = torch.clamp(sigma_0_map, 0, 1)
        #     mu_1_map = torch.clamp(mu_1_map, 0, 1)
        #     mu_0_map = torch.clamp(mu_0_map, 0, 1)

        ret_list = self.rendering(x.shape, attenuation_medium_map, mu_0_map, mu_1_map, sigma_0_map, z_vals=z_vals, refl_map=refl_map, boundary_map=boundary_map)

        intensity_map  = ret_list[0]

        x = torch.rot90(x, k=3, dims=[2, 3])
        intensity_map = torch.rot90(intensity_map, k=3, dims=[2, 3])        
        
        inverse_grid = self.inverse_grid.repeat(repeats)
        
        x_o = F.grid_sample(x.float(), inverse_grid, mode='nearest', padding_mode='zeros', align_corners=True).long()
        intensity_map = F.grid_sample(intensity_map.float(), inverse_grid, mode='bilinear', padding_mode='zeros', align_corners=True)
        
        mask_fan = self.mask_fan.repeat(repeats)
        intensity_map = intensity_map * mask_fan
        intensity_map_s = self.smooth(intensity_map)
        intensity_map[mask_fan==0] = intensity_map_s[mask_fan==0]

        return intensity_map, x, x_o, intensity_map_s, attenuation_medium_map, mu_0_map, mu_1_map, sigma_0_map, acoustic_imped_map, boundary_map, shifted_arr
    
    def configure_optimizers(self):        
        
        optimizer = optim.AdamW(self.parameters(),
                                lr=self.hparams.lr,
                                weight_decay=self.hparams.weight_decay)
        
        return optimizer
    

    def training_step(self, train_batch, batch_idx):
        seg = train_batch['seg']
        img = train_batch['img']

        fake_us = self(seg)

        repeats = [1,]*len(train_batch.shape)
        repeats[0] = train_batch.shape[0]
        mask_fan = self.mask_fan.repeat(repeats)
        img = mask_fan*img

        loss = self.loss(fake_us, img)

        self.log("loss", loss)

        return loss
        
    
    def validation_step(self, val_batch, batch_idx):        
        seg = val_batch['seg']
        img = val_batch['img']

        fake_us = self(seg)

        repeats = [1,]*len(val_batch.shape)
        repeats[0] = val_batch.shape[0]
        mask_fan = self.mask_fan.repeat(repeats)
        img = mask_fan*img

        val_loss = self.loss(fake_us, img)

        self.log("val_loss", val_loss, sync_dist=True)

In [ ]:
# fn = "/mnt/raid/C1_ML_Analysis/source/blender/simulated_data_export/FAM-202-1960-2_mesh_sampling/AC/98.nrrd"
fn = "/mnt/raid/C1_ML_Analysis/source/blender/simulated_data_export/FAM-202-1960-2_mesh_sampling/BPD/119.nrrd"
img = sitk.ReadImage(fn)
img1_np = sitk.GetArrayFromImage(img).astype(np.int32)
px.imshow(img1_np[0])

In [ ]:
fn2 = "/mnt/raid/C1_ML_Analysis/source/blender/simulated_data_export/FAM-202-1960-2_mesh_sampling/AC/40.nrrd"
img2 = sitk.ReadImage(fn2)
img2_np = sitk.GetArrayFromImage(img2).astype(np.int32)
px.imshow(img2_np[0])

In [ ]:
# fn_must = "/mnt/raid/C1_ML_Analysis/source/blender/simulated_data_export/FAM-202-1960-2_mesh_diffusor_sampling_must/BPD/119.nrrd"
fn_must = "/mnt/raid/C1_ML_Analysis/source/blender/simulated_data_export/FAM-202-1960-2_mesh_diffusor_sampling_must/AC/40.nrrd"
img_must_np = sitk.GetArrayFromImage(sitk.ReadImage(fn_must)).astype(np.int32)


acoustic_params_fn = '/mnt/raid/C1_ML_Analysis/source/blender/simulated_data_export/FAM-202-1960-2_mesh_acoustic_params.csv'
# acoustic_params_df = pd.read_csv('/mnt/raid/C1_ML_Analysis/source/blender/simulated_data_export/FAM-202-1960-2_mesh_acoustic_params.csv')
us_render = UltrasoundRendering(num_labels=330, grid_w=256, grid_h=256, center_x=128.0, center_y=-2.5, r1=40.0, r2=250.0, theta=np.pi/6, alpha_coeff_boundary_map=0.1, beta_coeff_scattering=10, tgc=8)
us_render.init_params(pd.read_csv(acoustic_params_fn))
# fake_us = us_render(torch.randint(low=0, high=10, size=(2, 1, 256, 256)))

t = torch.tensor(np.stack([img1_np, img2_np])).to(torch.long)
# t = F.pad(t, (10,10,10,10), mode='constant', value=0)
# t[:,:,0:10,:] = 0
# t[:,:,-10:,:] = 0
# t[:,:,:,0:10] = 0
# t[:,:,:,-10:] = 0

# t = transforms.functional.affine(t, angle=0, scale=1, shear=0, translate=[-5, 0])

fake_us = us_render(t.permute(0, 1, 3, 2))
# 
# fake_us = us_render(t)

In [ ]:
fig = go.Figure()
# fig.add_trace(go.Heatmap(z=np.flip(img_must_np, axis=0), colorscale='gray', showscale=False))
fig.add_trace(go.Heatmap(z=np.flip(fake_us[1][1].detach().numpy().squeeze(), axis=0), colorscale='jet', opacity=1.0))
# fig.add_trace(go.Heatmap(z=np.flip(fake_us[1][0].detach().numpy().squeeze(), axis=0), colorscale='turbo', opacity=0.25))

fig.update_layout(
            autosize=False,
            width=860,
            height=860
        )

fig

In [ ]:
fig = go.Figure()
# fig.add_trace(go.Heatmap(z=np.flip(img_must_np, axis=0), colorscale='gray', showscale=False))
fig.add_trace(go.Heatmap(z=np.flip(fake_us[2][1].detach().numpy().squeeze(), axis=0), colorscale='jet', opacity=1.0))
# fig.add_trace(go.Heatmap(z=np.flip(fake_us[1][0].detach().numpy().squeeze(), axis=0), colorscale='turbo', opacity=0.25))

fig.update_layout(
            autosize=False,
            width=860,
            height=860
        )

fig

In [ ]:
fn_must_2 = "/mnt/raid/C1_ML_Analysis/source/blender/simulated_data_export/FAM-202-1960-2_mesh_diffusor_sampling/AC/40.nrrd"
img_must_2_np = sitk.GetArrayFromImage(sitk.ReadImage(fn_must_2)).squeeze().transpose()

fig = go.Figure()
fig.add_trace(go.Heatmap(z=np.flip(img2_np[0].transpose(), axis=0), colorscale='portland', opacity=0.5))
fig.add_trace(go.Heatmap(z=np.flip(fake_us[0][1].detach().numpy().squeeze(), axis=0), colorscale='rainbow', opacity=0.5, showscale=False))
# fig.add_trace(go.Heatmap(z=np.flip(fake_us[2][1].detach().numpy().squeeze(), axis=0), colorscale='blues', opacity=0.5, showscale=False))

fig.update_layout(
            autosize=False,
            width=860,
            height=860
        )

fig

In [ ]:
class LotusDataset(Dataset):
    def __init__(self, df, mount_point = "./", img_column="img_path", seg_column=None):
        self.df = df
        self.mount_point = mount_point
        self.img_column = img_column
        self.seg_column = seg_column
        
        self.loader = LoadImaged(keys=[self.img_column, self.seg_column])

    def __len__(self):
        return len(self.df.index)

    def __getitem__(self, idx):
        
        img_path = os.path.join(self.mount_point, self.df.iloc[idx][self.img_column])
        seg_path = os.path.join(self.mount_point, self.df.iloc[idx][self.seg_column])

        d = {self.img_column: img_path, self.seg_column: seg_path}
        
        return self.loader(d)
    
class LotusDataModule(pl.LightningDataModule):
    def __init__(self, df_train, df_val, df_test, mount_point="./", batch_size=256, num_workers=4, img_column="img_path", seg_column="seg_path", train_transform=None, valid_transform=None, test_transform=None, drop_last=False):
        super().__init__()

        self.df_train = df_train
        self.df_val = df_val
        self.df_test = df_test
        self.mount_point = mount_point
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.img_column = img_column        
        self.seg_column = seg_column        
        self.train_transform = train_transform
        self.valid_transform = valid_transform
        self.test_transform = test_transform
        self.drop_last=drop_last        

    def setup(self, stage=None):

        # Assign train/val datasets for use in dataloaders
        self.train_ds = monai.data.Dataset(data=LotusDataset(self.df_train, self.mount_point, img_column=self.img_column, seg_column=self.seg_column), transform=self.train_transform)
        self.val_ds = monai.data.Dataset(data=LotusDataset(self.df_val, self.mount_point, img_column=self.img_column, seg_column=self.seg_column), transform=self.valid_transform)
        self.test_ds = monai.data.Dataset(data=LotusDataset(self.df_test, self.mount_point, img_column=self.img_column, seg_column=self.seg_column), transform=self.test_transform)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, num_workers=self.num_workers, persistent_workers=True, pin_memory=True, drop_last=self.drop_last, shuffle=True, prefetch_factor=2)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.batch_size, num_workers=self.num_workers, persistent_workers=False, pin_memory=True, drop_last=self.drop_last)

    def test_dataloader(self):
        return DataLoader(self.test_ds, batch_size=self.batch_size, num_workers=self.num_workers, persistent_workers=True, pin_memory=True, drop_last=self.drop_last)
    
class LotusTrainTransforms:
    def __init__(self, height: int = 256):

        # image augmentation functions
        self.train_transform = Compose(
            [   
                EnsureChannelFirstd(keys=['img_path', 'seg_path'], channel_dim=-1),
                ToTensord(keys=['img_path', 'seg_path']),
                RandRotated(keys=['img_path', 'seg_path'], range_x=math.pi, mode=['bilinear', 'nearest'])
            ]
        )

    def __call__(self, inp):
        return self.train_transform(inp)  
    
class LotusEvalTransforms:
    def __init__(self, height: int = 256):

        # image augmentation functions
        self.eval_transform = Compose(
            [   
                EnsureChannelFirstd(keys=['img_path', 'seg_path'], channel_dim=-1),
                ToTensord(keys=['img_path', 'seg_path'])
            ]
        )

    def __call__(self, inp):
        return self.eval_transform(inp)  

In [ ]:
df_train = pd.read_csv('/mnt/raid/C1_ML_Analysis/source/blender/simulated_data_export/CSV_files/label_diffusor_train_train.csv')
df_val = pd.read_csv('/mnt/raid/C1_ML_Analysis/source/blender/simulated_data_export/CSV_files/label_diffusor_train_test.csv')
df_test = pd.read_csv('/mnt/raid/C1_ML_Analysis/source/blender/simulated_data_export/CSV_files/label_diffusor_test.csv')

train_transform = LotusTrainTransforms()
valid_transform = LotusEvalTransforms()

dm = LotusDataModule(df_train, df_val, df_test, mount_point='/mnt/raid/C1_ML_Analysis/source/blender/simulated_data_export', batch_size=4, num_workers=1, img_column="img_path", seg_column="seg_path", train_transform=train_transform, valid_transform=valid_transform)
dm.setup()

In [ ]:
train_dl = dm.train_dataloader()


In [ ]:
# for b in train_dl:
#     print(b['img_path'].shape, b['seg_path'].shape)

In [ ]:
from monai.networks.blocks import (
    MLPBlock
)

In [ ]:
MLPBlock(hidden_size=256, mlp_dim=256)

In [ ]:
monai.networks.nets.Unet(spatial_dims=2,
    in_channels=2,
    out_channels=1,
    channels=(4, 8, 16),
    strides=(2, 2),
    num_res_units=2)

In [ ]:
class TimeDistributed(nn.Module):
    def __init__(self, module, time_dim=1):
        super(TimeDistributed, self).__init__()
        self.module = module
        self.time_dim = time_dim
 
    def forward(self, input_seq):
        assert len(input_seq.size()) > 2
 
        # reshape input data --> (samples * timesteps, input_size)
        # squash timesteps

        size = list(input_seq.size())

        batch_size = size[0]
        time_steps = size.pop(self.time_dim)

        size_reshape = [batch_size*time_steps] + list(size[1:])
        reshaped_input = input_seq.contiguous().view(size_reshape)
 
        output = self.module(reshaped_input)
        
        output_size = [batch_size] + list(output.size())[1:]
        output_size.insert(self.time_dim, time_steps)
        
        output = output.contiguous().view(output_size)

        return output
    
time_d = TimeDistributed(nn.Conv1d(in_channels=1, out_channels=1, kernel_size=33, padding='same'), time_dim=2)
time_d(torch.rand(2, 1, 256, 256)).shape